In [24]:
""""
Summarize fuel treatment acres within fire perimeters
Maxwell.Cook@colostate.edu
"""

import os, sys

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from __functions import *  # imports all custom functions

# local data directories
datadir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/data/'
projdir = os.path.dirname(os.getcwd())
print(projdir)

print("Ready !")

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation
Ready !


In [25]:
# load the incidents / perimeters
mtbs = os.path.join(projdir, 'data/spatial/wf_incidents_2014_2022_mtbs_perims.gpkg')
mtbs = gpd.read_file(mtbs)
print(f"Processing for {len(mtbs)} fires.")
print(f"CRS: {mtbs.crs}")
# clean the dataframe just a bit for processing
mtbs = mtbs[['Event_ID','Incid_Name','Ig_Date','BurnBndAc','geometry']]

Processing for 3494 fires.
CRS: EPSG:4326


In [26]:
# load the fuel treatment data
fp = os.path.join(projdir, 'data/spatial/treatments/twig_treatments_1km.gpkg')
trts = gpd.read_file(fp)
# convert date fields and extract the year
trts['treatment_date'] = pd.to_datetime(trts['treatment_date'], unit='ms', errors='coerce')
trts['actual_completion_date'] = pd.to_datetime(trts['actual_completion_date'], unit='ms', errors='coerce')
trts['year_comp'] = trts['actual_completion_date'].dt.year
# check on the structure
print(f"Columns:\n{trts.columns}")
print(f"CRS: {trts.crs}")
trts[['Event_ID', 'unique_id', 'treatment_date', 'actual_completion_date', 'year_comp', 'type', 'acres']].head()

Columns:
Index(['type', 'date_source', 'identifier_database', 'fund_source', 'state',
       'SHAPE__Area', 'unique_id', 'actual_completion_date', 'agency',
       'date_current', 'twig_category', 'treatment_date', 'name', 'acres',
       'category', 'SHAPE__Length', 'objectid', 'index_right', 'Event_ID',
       'activity', 'total_cost', 'cost_per_uom', 'fund_code', 'uom', 'method',
       'equipment', 'activity_code', 'error', 'geometry', 'year_comp'],
      dtype='object')
CRS: EPSG:3857


,Event_ID,unique_id,treatment_date,actual_completion_date,year_comp,type,acres
0,FL2632608099820220328,124457-6327377,2021-02-20,2021-02-20,2021.0,Broadcast Burn,512.755928
1,FL2632608099820220328,124454-6327377,2021-02-20,2021-02-20,2021.0,Broadcast Burn,270.601716
2,FL2632608099820220328,124466-6327385,2021-03-26,2021-03-26,2021.0,Broadcast Burn,92.101353
3,FL2632608099820220328,124468-6327385,2021-03-26,2021-03-26,2021.0,Broadcast Burn,94.878239
4,FL2632608099820220328,132231-6327393,2022-02-28,2022-02-28,2022.0,Broadcast Burn,267.597887


In [27]:
trts['type'].unique()

array(['Broadcast Burn', 'Machine Pile Burn', 'Thinning', 'Fire Use',
       None, 'Biomass Removal', 'Mowing', 'N/A', 'Chipping',
       'Mastication', 'Machine Pile', 'Lop and Scatter', 'Hand Pile',
       'Hand Pile Burn', 'Chemical', 'Crushing', 'Grazing', 'Seeding',
       'Jackpot Burn', 'Mastication/Mowing', 'Biological',
       'Chemical Treatment', 'Preparation'], dtype=object)

In [28]:
# subset to our treatments of interest
trts_keep = [
    'Thinning', # this includes both hand and mechanical thinning
    'Machine Pile', 'Hand Pile',
    'Mastication', 'Lop and Scatter',
    'Broadcast Burn', 'Jackpot Burn', 'Fire Use',
    'Machine Pile Burn', 'Hand Pile Burn',
    'Biomass Removal'
]

# filter our dissolved data
trts_ = trts[trts['type'].isin(trts_keep)].copy()
print(trts_['type'].unique())

['Broadcast Burn' 'Machine Pile Burn' 'Thinning' 'Fire Use'
 'Biomass Removal' 'Mastication' 'Machine Pile' 'Lop and Scatter'
 'Hand Pile' 'Hand Pile Burn' 'Jackpot Burn']


In [29]:
# crop treatments to the fire perimeters
# buffer perimeters 1km
print("Buffering fire perimeters ...")
buffer = mtbs.copy().to_crs(trts.crs) # match CRS to treatments
buffer['geometry'] = buffer.geometry.buffer(1000)
buffer['fire_buffer_acres'] = buffer.geometry.area / 4046.86 # recalculate mtbs acres with buffer
buffer = buffer.to_crs(trts.crs) # now match the treatment CRS

# loop through fires, crop treatment data to bounds
trt_clips = [] # to store the results
for idx, fire_row in tqdm(buffer.iterrows(), total=buffer.shape[0], desc='Processing Fires'):
    fire_id = fire_row['Event_ID']
    fire_acres = fire_row['fire_buffer_acres']
    fire_geom = fire_row['geometry']

    # select overlapping treatments
    trts_fire = trts_[trts_['Event_ID'] == fire_id]

    if not trts_fire.empty:
        # Clip treatments to the fire perimeter buffer
        clipped = gpd.clip(trts_fire, fire_geom)
        clipped['fire_buffer_acres'] = fire_acres
        trt_clips.append(clipped)
        del clipped, trts_fire

# Combine all clipped results into one GeoDataFrame
trts_fire = gpd.GeoDataFrame(pd.concat(trt_clips, ignore_index=True), crs=trts.crs)
del trt_clips

# Recalculate area
trts_fire['gis_acres'] = trts_fire.geometry.area / 4046.86
trts_fire.head()

Buffering fire perimeters ...


Processing Fires:   0%|          | 0/3494 [00:00<?, ?it/s]

,type,date_source,identifier_database,fund_source,state,SHAPE__Area,unique_id,actual_completion_date,agency,date_current,...,fund_code,uom,method,equipment,activity_code,error,geometry,year_comp,fire_buffer_acres,gis_acres
0,Broadcast Burn,act_comp_dt,NFPORS,No Funding Code,FL,7.562210e+06,124458-6327377,2021-02-20,BIA,1.633944e+12,...,None,None,None,None,None,None,"POLYGON ((-9025720.879 3033185.453, -9025430.0...",2021.0,12509.243222,483.840992
1,Broadcast Burn,act_comp_dt,NFPORS,No Funding Code,FL,4.932693e+06,57881-6298261,2016-07-30,BIA,1.497030e+12,...,None,None,None,None,None,None,"MULTIPOLYGON (((-9019761.687 3034134.1, -90193...",2016.0,12509.243222,46.451566
2,Broadcast Burn,act_comp_dt,NFPORS,No Funding Code,FL,9.956538e+06,102593-6311388,2018-03-09,BIA,1.537176e+12,...,None,None,None,None,None,None,"POLYGON ((-9023681.363 3032145.288, -9023658.7...",2018.0,12509.243222,1128.538188
3,Broadcast Burn,act_comp_dt,NFPORS,No Funding Code,FL,6.957308e+06,132238-6327392,2022-01-04,BIA,1.665515e+12,...,None,None,None,None,None,None,"POLYGON ((-9026895.363 3034574.504, -9026802.0...",2022.0,12509.243222,3.283526
4,Broadcast Burn,act_comp_dt,NFPORS,No Funding Code,FL,3.856704e+05,102592-6311388,2018-03-09,BIA,1.537176e+12,...,None,None,None,None,None,None,"MULTIPOLYGON (((-9025737.73 3033428.331, -9025...",2018.0,12509.243222,10.634004


In [30]:
# Flatten (dissolve) by Fire ID, Treatment Type, and Year Completed
flat = trts_fire[trts_fire['type'] != 'None'].dissolve(by=['Event_ID','type','year_comp']).reset_index()
flat['type'].unique() # check what treatment types we're working with
flat['trt_gis_acres'] = flat.geometry.area / 4046.86 # recalculate the GIS acres
# keep only a few attributes ...
flat_c = flat[['Event_ID', 'type', 'year_comp', 'trt_gis_acres', 'fire_buffer_acres', 'geometry']].copy()
print(flat_c['type'].unique())

['Broadcast Burn' 'Biomass Removal' 'Lop and Scatter' 'Machine Pile'
 'Thinning' 'Fire Use' 'Machine Pile Burn' 'Mastication' 'Hand Pile Burn'
 'Jackpot Burn' 'Hand Pile']


In [31]:
# now create the single dissolved layer by fire ID
# add the treatment years as a list
flat_c_fire = (
    flat_c
    .dissolve(
        by=["Event_ID", "type"], # fire/treatment type
        aggfunc={
            "year_comp": lambda x: sorted(set(x.dropna().astype(int))), # list of treatment years
            "fire_buffer_acres": "first"   # keep the fire acres as a column in the output
        }
    )
    .reset_index()
)

# clip again to fire buffers
flat_c_fire = gpd.clip(flat_c_fire, buffer)
# recalculate the treatment acres
flat_c_fire['treatment_acres'] = flat_c_fire.geometry.area / 4046.86
# calculate the percent treated by treatment type
flat_c_fire['fire_pct_treated'] = (flat_c_fire['treatment_acres'] / flat_c_fire['fire_buffer_acres']) * 100
print(len(flat_c_fire))
flat_c_fire.head()

2729


,Event_ID,type,geometry,year_comp,fire_buffer_acres,treatment_acres,fire_pct_treated
1049,FL2523008087020180726,Broadcast Burn,"POLYGON ((-8999282.936 2902955.601, -8999307.5...","[2009, 2012]",3964.692412,1843.279170,46.492363
1050,FL2563608053120200324,Broadcast Burn,"POLYGON ((-8968241.202 2954250.174, -8968231.1...","[2011, 2014, 2015, 2020]",3989.602204,1865.519261,46.759531
1051,FL2563608053120200324,Lop and Scatter,"POLYGON ((-8966479.533 2954708.065, -8966465.8...",[2016],3989.602204,471.679634,11.822723
1053,FL2574708088120200507,Broadcast Burn,"MULTIPOLYGON (((-9002552.059 2954850.819, -900...","[2014, 2015]",49194.064456,2244.811510,4.563176
1052,FL2565208050520200419,Lop and Scatter,"MULTIPOLYGON (((-8962522.078 2959951.049, -896...","[2016, 2018]",5940.621994,1117.523668,18.811560


In [32]:
# Join in the unique number of treatments by type
# Count the unique treatments of each type
n_trts = (
    trts_fire
    .groupby(["Event_ID", "type"])
    .agg(n_treatments=("year_comp", "nunique"))
    .reset_index()
)
# join to the dataframe
flat_c_fire_ = pd.merge(flat_c_fire, n_trts, on=["Event_ID", "type"], how="left")
# tidy up some of the columns
flat_c_fire_ = flat_c_fire_[[
    'Event_ID', 'type', 'treatment_acres',
    'fire_buffer_acres', 'fire_pct_treated',
    'year_comp', 'n_treatments', 'geometry'
]]
flat_c_fire_.rename(columns={
    'type': 'treatment_type',
    'year_comp': 'treatments_years'
}, inplace=True)
print(flat_c_fire_['fire_pct_treated'].describe())
flat_c_fire_.head()

count    2729.000000
mean        6.673918
std        15.783787
min         0.000024
25%         0.192611
50%         0.987964
75%         4.517995
max       100.000000
Name: fire_pct_treated, dtype: float64


,Event_ID,treatment_type,treatment_acres,fire_buffer_acres,fire_pct_treated,treatments_years,n_treatments,geometry
0,FL2523008087020180726,Broadcast Burn,1843.279170,3964.692412,46.492363,"[2009, 2012]",2,"POLYGON ((-8999282.936 2902955.601, -8999307.5..."
1,FL2563608053120200324,Broadcast Burn,1865.519261,3989.602204,46.759531,"[2011, 2014, 2015, 2020]",4,"POLYGON ((-8968241.202 2954250.174, -8968231.1..."
2,FL2563608053120200324,Lop and Scatter,471.679634,3989.602204,11.822723,[2016],1,"POLYGON ((-8966479.533 2954708.065, -8966465.8..."
3,FL2574708088120200507,Broadcast Burn,2244.811510,49194.064456,4.563176,"[2014, 2015]",2,"MULTIPOLYGON (((-9002552.059 2954850.819, -900..."
4,FL2565208050520200419,Lop and Scatter,1117.523668,5940.621994,18.811560,"[2016, 2018]",2,"MULTIPOLYGON (((-8962522.078 2959951.049, -896..."


In [33]:
# Identify "Thin+Rx Fire" category
thin = flat_c_fire_[flat_c_fire_['treatment_type']=="Thinning"]
rx = flat_c_fire_[flat_c_fire_['treatment_type']=="Broadcast Burn"]

# Perform an intersection of these treatment types
thin_rx = gpd.overlay(thin, rx, how="intersection", keep_geom_type=False)
# make sure we are working within the same fire event
thin_rx = thin_rx[thin_rx['Event_ID_1'] == thin_rx['Event_ID_2']]
# rename the ID column and the fire acres
thin_rx = thin_rx.rename(columns={"Event_ID_1": "Event_ID"})
thin_rx["fire_buffer_acres"] = thin_rx["fire_buffer_acres_1"]

# Recalculate overlap acres and percent
thin_rx["treatment_acres"] = thin_rx.geometry.area / 4046.86
thin_rx["fire_pct_treated"] = (thin_rx["treatment_acres"] / thin_rx["fire_buffer_acres"]) * 100

# Label new treatment type
thin_rx["treatment_type"] = "Thin + Rx Fire"

# Keep consistent schema with main table
thin_rx = thin_rx[[
    "Event_ID","treatment_type","treatment_acres",
    "fire_buffer_acres","fire_pct_treated","geometry"
]]
thin_rx.head()

,Event_ID,treatment_type,treatment_acres,fire_buffer_acres,fire_pct_treated,geometry
0,FL3015508446920200817,Thin + Rx Fire,18.884879,4010.680685,0.470865,"POLYGON ((-9404094.752 3525591.912, -9404101.7..."
1,FL3020808495520160622,Thin + Rx Fire,966.979984,5585.863166,17.311201,GEOMETRYCOLLECTION (POLYGON ((-9457060.213 352...
2,MS3054708884920220321,Thin + Rx Fire,12.596237,4637.401582,0.271623,"MULTIPOLYGON (((-9893561.771 3573800.921, -989..."
3,FL3031908451920190528,Thin + Rx Fire,74.220894,3546.877142,2.092570,"POLYGON ((-9406006.456 3544790.786, -9406017.7..."
4,FL3033808442820180309,Thin + Rx Fire,41.879271,14205.166660,0.294817,"POLYGON ((-9403447.134 3546210.655, -9403435.1..."


In [34]:
# merge the thin+rx fire back
flat_c_fire_int = pd.concat([flat_c_fire_, thin_rx], ignore_index=True)
flat_c_fire_int['treatment_type'].unique()

array(['Broadcast Burn', 'Lop and Scatter', 'Mastication',
       'Hand Pile Burn', 'Machine Pile Burn', 'Fire Use',
       'Biomass Removal', 'Machine Pile', 'Thinning', 'Jackpot Burn',
       'Hand Pile', 'Thin + Rx Fire'], dtype=object)

In [35]:
# save this long format table
out_fp = os.path.join(projdir, 'data/tabular/MTBS_TWIG_summary_long-format.csv')
flat_c_fire_int.to_csv(out_fp, index=False)
print(f"Saved to: {out_fp}")

Saved to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/tabular/MTBS_TWIG_summary_long-format.csv


In [36]:
# make the table wide to easily join with model data
flat_c_fire_w = (
    flat_c_fire_int
    .pivot_table(
        index="Event_ID",
        columns="treatment_type",
        values="fire_pct_treated",   # proportion of fire treated
        aggfunc="sum",               # if multiple rows per fire/type, sum them
        fill_value=0
    )
    .reset_index()
)
# merge back the other attributes
print(len(flat_c_fire_int))
print(len(flat_c_fire_w))
flat_c_fire_w.head(10)

2945
1042


treatment_type,Event_ID,Biomass Removal,Broadcast Burn,Fire Use,Hand Pile,Hand Pile Burn,Jackpot Burn,Lop and Scatter,Machine Pile,Machine Pile Burn,Mastication,Thin + Rx Fire,Thinning
0,AL3274708707120221010,0.000000,10.773493,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
1,AL3277908693520190402,0.000000,29.541672,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
2,AL3317108611720161113,1.881704,10.885592,0.000000,0.0,0.0,0.0,1.181918,0.192194,0.0,0.0,0.635470,0.695732
3,AL3326608609720161012,4.963832,65.222398,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,10.172570,10.370532
4,AL3337208592120161029,0.709634,5.696312,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.026757,0.028901
5,AL3345508578420211119,2.952686,6.159663,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,4.538344,8.099145
6,AL3346508586820161128,3.252638,81.134067,3.456146,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,2.673176,2.825038
7,AL3435008732320220213,0.000000,49.024224,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,2.534552,2.534559
8,AR3435309394120211128,0.000000,5.763829,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
9,AR3483509369420180322,5.382286,52.168649,0.000000,0.0,0.0,0.0,0.000000,15.345585,0.0,0.0,17.078733,20.629323


In [37]:
# check on the cameron peak fire as an example
flat_c_fire_w[flat_c_fire_w['Event_ID']=='CO4060910587920200813']

treatment_type,Event_ID,Biomass Removal,Broadcast Burn,Fire Use,Hand Pile,Hand Pile Burn,Jackpot Burn,Lop and Scatter,Machine Pile,Machine Pile Burn,Mastication,Thin + Rx Fire,Thinning
331,CO4060910587920200813,5.063942,0.977068,0.0,0.0,0.0,0.0,0.013526,0.819024,1.199568,0.0,0.074371,0.655455


### Tidy the previous burn data

In [22]:
fp = os.path.join(projdir, 'data/spatial/treatments/nifc_perimeters_1km.gpkg')
fires = gpd.read_file(fp)
fires.head()

,OBJECTID,IRWINID,FORID,INCIDENT,GIS_ACRES,UNQE_FIRE_ID,DATE_CUR,FIRE_YEAR_INT,UNIT_ID,POO_RESP_I,...,GEO_ID,SOURCE,AGENCY,FIRE_YEAR,GlobalID,Shape__Area,Shape__Length,index_right,Event_ID,geometry
0,102308,None,{665F1DE8-9030-4934-B81A-8061939B6413},Cowbone,1046.78900,2015-FLBCP-000077,20210121174400,2015,FLBCP,None,...,{ED6CC41D-A0FC-409B-BA01-67CE27E3D914},NPS,NPS,2015,873cea06-3311-4f89-9036-4aef26e760fa,5.291022e+06,10780.052327,3,FL2632608099820220328,"MULTIPOLYGON (((-9023119.372 3034496.954, -902..."
1,105412,None,{FD166580-6CE0-4615-B738-F7599FF2FC45},Dumpsite,10.54813,2017-FLSEA-000012,20170626,2017,FLSEA,None,...,2,BIA,BIA,2017,2253ece6-c131-4afb-b28e-d99acd7760f2,5.333058e+04,919.275308,3,FL2632608099820220328,"MULTIPOLYGON (((-9019785.306 3036074.988, -901..."
2,107110,{07627D47-33EE-40C9-8773-DF8DDCF4D748},{7361E0F1-432B-4CFE-B1DA-A87B336BE5F0},Interceptor,2904.52000,2022-FLSEA-000494,202303141513,2022,FLSEA,FLSEA,...,{2D450A22-26E2-41BB-8A26-BE8C8B195925},WFIGS,BIA,2022,0ad8b595-cf7d-4ef1-8091-8f2cb7d012c7,1.468353e+07,70838.195952,3,FL2632608099820220328,"MULTIPOLYGON (((-9024207.31 3033338.248, -9024..."
3,109621,{5C47BBB7-0AE3-4CE6-8A02-A17D0C0106E3},{F6591631-E260-422E-AB48-5AA65482163C},HANCOCK CREEK,1177.38000,2022-NCNCF-220167,20230314151342,2022,NCNCF,NCNCF,...,{9b804ff4-e581-4f56-8c32-a4318372c814},WFIGS,USFS,2022,d6e6de07-d974-4388-8505-9e807c3dbfd8,7.104382e+06,13047.198834,6,NC3492707684920220414,"MULTIPOLYGON (((-8552172.77 4152569.793, -8553..."
4,28351,{4CE00061-B897-49DC-8839-D08E040FF65D},{0CFFBF9F-BC02-4779-91B0-55229071EB3B},CASTLE,170648.00000,2020-CASQF-002541,20201002234043,2020,CASQF,CASQF,...,{557026C3-5C2F-4473-8500-F29DC356E586},USFS,USFS,2020,c1d0e295-0944-469a-b6a1-40c73f7765a8,1.063136e+09,897716.913345,21,CA3633311865120220803,"MULTIPOLYGON (((-13206571.765 4342942.979, -13..."


In [23]:
# Dissolve by Event ID
fires_flat = fires.dissolve(by=['Event_ID']).reset_index()
fires_union = gpd.GeoDataFrame(geometry=[fires_flat.union_all()], crs=fires_flat.crs)  # dissolve into one shape
# Intersect with MTBS perimeters (buffer)
intersections = gpd.overlay(buffer[['Event_ID','geometry']], fires_union, how="intersection")
intersections['overlap_acres'] = intersections.geometry.area / 4046.86
# Summarize
summary = intersections.groupby("Event_ID", as_index=False)['overlap_acres'].sum()
summary = summary.merge(buffer[['Event_ID','fire_buffer_acres']], on='Event_ID', how='left')
summary['Wildfire'] = summary['overlap_acres'] / summary['fire_buffer_acres']
print(summary['Wildfire'].describe())
summary.head()

count    1.554000e+03
mean     5.567314e-01
std      2.438669e-01
min      5.295864e-08
25%      3.913878e-01
50%      5.642679e-01
75%      7.421208e-01
max      1.000000e+00
Name: Wildfire, dtype: float64


,Event_ID,overlap_acres,fire_buffer_acres,Wildfire
0,AL3274708707120221010,998.110934,3947.103378,0.252872
1,AL3277908693520190402,3696.951891,7447.147199,0.496425
2,AL3317108611720161113,1548.816858,5345.044578,0.289767
3,AL3326608609720161012,980.640050,4502.264460,0.217810
4,AL3337208592120161029,1462.218200,5965.871209,0.245097


In [38]:
# merge with the treatment data
trts_w_fire = pd.merge(flat_c_fire_w, summary[['Event_ID','Wildfire']], on=["Event_ID"], how="left")
trts_w_fire.head()

,Event_ID,Biomass Removal,Broadcast Burn,Fire Use,Hand Pile,Hand Pile Burn,Jackpot Burn,Lop and Scatter,Machine Pile,Machine Pile Burn,Mastication,Thin + Rx Fire,Thinning,Wildfire
0,AL3274708707120221010,0.000000,10.773493,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.252872
1,AL3277908693520190402,0.000000,29.541672,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.496425
2,AL3317108611720161113,1.881704,10.885592,0.0,0.0,0.0,0.0,1.181918,0.192194,0.0,0.0,0.635470,0.695732,0.289767
3,AL3326608609720161012,4.963832,65.222398,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,10.172570,10.370532,0.217810
4,AL3337208592120161029,0.709634,5.696312,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.026757,0.028901,0.245097


In [39]:
# save the wide format table (just treatment percentages)
out_fp = os.path.join(projdir, 'data/tabular/MTBS_TWIG_summary_wide-format.csv')
trts_w_fire.to_csv(out_fp, index=False)
print(f"Saved to: {out_fp}")

Saved to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/tabular/MTBS_TWIG_summary_wide-format.csv
